# Cynbit AI – Lead Classification & ML Pipeline Training

**Role:** AI/ML Engineer Intern Technical Assessment  
**Company:** Cynbit Technologies  
**Project:** Intelligent Lead Qualification & Automation Agent  

This notebook documents the end-to-end Machine Learning pipeline for classifying inbound business inquiries into 6 core service categories:
1. `AI/ML Development`
2. `Web Development`
3. `Mobile App Development`
4. `Cloud & DevOps`
5. `Business Automation`
6. `Other`

---

## 1. Environment Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load training dataset
data_path = os.path.join('..', 'data', 'training_leads.csv')
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} inquiry samples.")
display(df.head())
print("\nClass Distribution:")
print(df['category'].value_counts())

## 2. Text Preprocessing & Cleaning

Standardizing raw user text: lowercase conversion, contraction expansion, non-alphanumeric filtering, and stopword removal.

In [ ]:
import sys
sys.path.append('..')
from app.ml.pipeline import clean_text

df['cleaned_text'] = df['inquiry_text'].apply(clean_text)
df[['inquiry_text', 'cleaned_text', 'category']].sample(5, random_state=42)

## 3. Stratified Train / Test Split

Preserving balanced class distributions across the 80% training set and 20% held-out evaluation set.

In [ ]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text'].tolist()
y = df['category'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Training set size: {len(X_train)} samples")
print(f"Test set size:     {len(X_test)} samples")

## 4. Feature Engineering: TF-IDF Vectorization

Extracting unigram and bigram term frequency-inverse document frequency features with sublinear scaling.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print(f"Fitted Vocabulary Size: {len(vectorizer.get_feature_names_out())} features")

## 5. Model Training & Comparison

Comparing Multi-class Logistic Regression with Multinomial Naive Bayes and Random Forest.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.5),
    'Random Forest (100 trees)': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression (L2, C=2.0)': LogisticRegression(C=2.0, max_iter=1000, class_weight='balanced', random_state=42)
}

results = []
for name, clf in models.items():
    clf.fit(X_train_vec, y_train)
    preds = clf.predict(X_test_vec)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='macro')
    results.append({'Model': name, 'Test Accuracy': acc, 'Macro F1': f1})

results_df = pd.DataFrame(results)
display(results_df)

## 6. Detailed Evaluation of the Production Classifier

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

best_model = models['Logistic Regression (L2, C=2.0)']
y_pred = best_model.predict(X_test_vec)

labels = sorted(list(set(y)))
print("=== CLASSIFICATION REPORT (Held-out 20% Test Set) ===")
print(classification_report(y_test, y_pred, labels=labels, zero_division=0))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[l.split()[0] for l in labels], yticklabels=[l.split()[0] for l in labels])
plt.title('Confusion Matrix on Test Inquiries')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 7. Model Interpretability: Top Predictive Features per Service

In [ ]:
feature_names = np.array(vectorizer.get_feature_names_out())
for i, label in enumerate(best_model.classes_):
    top_indices = np.argsort(best_model.coef_[i])[-6:][::-1]
    top_feats = feature_names[top_indices]
    top_weights = best_model.coef_[i][top_indices]
    print(f"\nCategory: {label}")
    for f, w in zip(top_feats, top_weights):
        print(f"  • {f:<20} (+{w:.3f})")

## 8. Serialization of Production Artifacts

In [ ]:
import joblib

os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/lead_classifier.joblib')
joblib.dump(vectorizer, '../models/tfidf_vectorizer.joblib')
print("Trained model and vectorizer saved to disk successfully.")